# Matrix Inversion Using ML

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import networkx as nx

import random
import time

## Printing and Display

In [ ]:
def print_matrix(M, precision=3):
  size = np.shape(M)[0]
  print("")
  for row in range(size):
    print(f'{[round(M[row,col], precision) for col in range(size)]}')

def reconstruct_array(u,s,v_t):
  return np.einsum("ij,j,jk->ik", u, s, v_t)

In [ ]:
def draw_graph(M): # graph the network, with forward edges in the lower triangular region
  size = np.shape(M)[0]
  nodes = [chr((char%26 + 65)) for char in range(size)] # up to 26 nodes before repeats
  edges = []
  for row in range(size):
    for col in range(size):
      if M[col,row] != 0:
        edges.append((nodes[row], nodes[col]))

  G = nx.DiGraph()
  G.add_edges_from(edges)
  pos = nx.circular_layout(G)
  plt.figure(figsize=(10, 8))
  nx.draw_networkx_nodes(G, pos, node_color='grey', node_size=500)
  nx.draw_networkx_labels(G, pos)
  nx.draw_networkx_edges(G, pos, edgelist=G.edges(), arrows=True)
  plt.show()

In [ ]:
def conditional_print(Q,M):
  if n < 20:
    draw_graph(Q)
    print_matrix(Q)
    print_matrix(M)
    G = get_transfer_function(Q) # the true transfer function of Q
    print_matrix(G)
    print_matrix(G - estimate_transfer_function(Q, 20)) # proof of concept of G = (I-Q)^-1 = I + Q + Q^2 + Q^3 + ...

In [ ]:
def visualize_dict(matrixDict):  #a way of visualizing the dictionary with the rows and columns
    print()
    for i in matrixDict:
        print(f"Column {i}:")
        for j in matrixDict[i]:
            print(f"row {j} --> {matrixDict[i][j]}")
        print()


In [ ]:
def print_rounded(vect, epsilon=10**-3):
  np.round(vect, decimals=int(np.log10(epsilon)))
  print(vect) # should be 0.0

In [ ]:
def print_errors_array(est, actual, name, time):
  print(f"  For the {name} method ({time:.4f} sec):")
  print(f"    Act L0:   {measure(est-actual, 0)}")
  print(f"    Avg L0:   {measure(est-actual, 0)/n}")
  print(f"    Avg L1:   {measure(est-actual, 1)/n}")
  print(f"    Avg L2:   {np.sqrt(measure(est-actual, 2)/n)}")
  print(f"    Avg Linf: {measure(est-actual, np.inf)}")

def print_errors_dict(est, actual, name, time):
  errors = {}
  for i in est.keys():
    if i not in actual.keys():
      errors[i] = np.abs(est[i])
    else:
      errors[i] = np.abs(est[i] - actual[i])
  for i in actual.keys():
    if i in actual.keys():
      continue
    else:
      errors[i] = np.abs(actual[i])
  array = []
  for i in errors.keys():
    if errors[i] != 0.0:
      array.append(errors[i])


  print(f"  For the {name} method ({time:.4f} sec):")
  print(f"    Act L0:   {measure(array, 0)}")
  print(f"    Avg L0:   {measure(array, 0)/n}")
  print(f"    Avg L1:   {measure(array, 1)/n}")
  print(f"    Avg L2:   {np.sqrt(measure(array, 2)/n)}")
  print(f"    Avg Linf: {measure(array, np.inf)}")

## Matrix Logic

In [ ]:
#@title all of this code is by Micah
def generateBLTmatrix(n,k,per=.75):  #creats a matrix with length and width n and the largest posible block along the diagonal is of length and width k
    matrix = np.empty((0,n), int)  # creates an empty matrix so np.append can be used

    if k > n:  #make sure that k is smaller then n
        raise ValueError

    def add_row(n, numk, timesused, per):  #function that adds a row onto the matrix
        array = np.array([])
        for i in range(timesused):
            numOfZeros = random.random()
            if numOfZeros < per:
                array = np.append(array, np.array([0]))
            else:
                array = np.append(array, np.array([random.random()]))
        for i in range(timesused, numk+timesused):
            array = np.append(array, np.array([random.random()]))
        for i in range(numk+timesused, n):
            array = np.append(array, np.array([0]))
        return array

    endLoopEarly = 0  #endloopearly is a variable that allows to break out of loop once it its the max number of lines
    numkadd = 0
    for i in range(n):
        if (numkadd + k) > n and numkadd != n:
            numk = random.randint(1,n-numkadd)
        else:
            numk = random.randint(1,k)
        for i in range(numkadd, numk+numkadd):
            matrix = np.vstack([matrix, add_row(n,numk,numkadd,per)])
            endLoopEarly += 1
        numkadd = numkadd + numk
        if endLoopEarly == n:
            break

    return matrix

def setTotalColumnValueToOne(matrix):  #Sets the sum of each column with index of the same row to 1
    for i in range(0, len(matrix)):
        sumOf = 0
        for j in matrix:
            sumOf = sumOf + j[i]
        for k in matrix:
            k[i] = k[i] / sumOf
    return matrix

In [ ]:
#@title generate an adjacency list (invertible matrix of size "size" with one to "num_out_edges" entries per column) with column sums "sum_of_each_column"
def generateSparseAdjacencyList(size, num_out_edges, sum_of_each_column):
  M = {}
  for col in range(size):
    M[col] = {col:1}

  for cycle in range(num_out_edges):
    # generate num_out_edges permutations of size numbers
    perm = np.random.permutation(size)

    # each permutation is a path from prev to next, so the entries in the matrix are M[col][row] == M[perm[curr], perm[next]]
    # give these entries random values from -1 to 1
    for index in range(size):
      M[perm[index]][perm[(index+1)%size]] = np.random.random()

  # normalize the columns after we are done to sum to sum_of_each_column
  for col in range(size):
    magnitude = np.sum(np.abs(list(M[col].values()))) / sum_of_each_column
    for row in M[col].keys():
      # Q needs to have spectral radius less than 1, not M
      M[col][row] = (row==col) - M[col][row] / magnitude

  return M

In [ ]:
#@title an adjacency list elementary unit vector (jth column of the identity)
def elem_vect(n, j):
  e_j = np.zeros((n,))
  e_j[j] = 1.0
  return e_j

In [ ]:
#@title convert an ndarray vector into an adjacency list
def vectToDict(vect):
  _dict = {}
  for i in range(n):
    _dict[i] = vect[i]
  return _dict

In [ ]:
#@title convert an ndarray matrix into an adjacency list
def matrixToDict(matrix):  #creates a dictionay with all of the columns as seperate dictionaries
    dictlist = {}
    length = matrix.shape[0]

    for col in range(length):
      if np.max(np.abs(M[:,col])) == 0:
        continue
      dictlist[col] = vectToDict(matrix[:, col])
    return dictlist

In [ ]:
#@title convert a dict to an ndarray
def dict_to_vect(vect, size):
  array = np.zeros((size,))
  for key in vect.keys():
    array[key] += vect[key]
  return array

def dict_to_mat(mat, size):
  array = np.zeros((size, size))
  for key in mat.keys():
    array[:,key] += dict_to_vect(mat[key], size)
  return array

In [ ]:
#@title Multiply a matrix[column, row] on the right by a column vector u
def dict_mat_mult(M, u, epsilon=0.0):
  y = {} # y = Mu
  FLOPs = 0
  for col in u.keys():
    if col not in M.keys():
      continue
    for row in M[col].keys():
      if row not in y.keys():
        y[row] = 0.0
      FLOPs += 1 # for multiplication
      y[row] += M[col][row] * u[col] # the column of M matches the row of u
      if abs(y[row]) <= epsilon:
        del y[row]
  return y, FLOPs

In [ ]:
def get_Q(M, row, col): # Q = I - M
  return (row == col) - M[col][row]

def mul_Q_left(M, vect, epsilon=0.0):
  y = {} # y = M * vect
  FLOPs = 0
  for col in vect.keys():
    for row in M[col].keys():
      if row not in y.keys():
        y[row] = 0.0
      y[row] += get_Q(M, row, col) * vect[col] # the column of M matches the row of vect
      FLOPs += 1
      if abs(y[row]) < epsilon:
        del y[row]
  return y, FLOPs

def add_vects(left, right):
  for row in right.keys():
    if row not in left.keys():
      left[row] = right[row]
    else:
      left[row] += right[row]
  return left, len(right.keys())

In [ ]:
#@title A method for measuring norms of an ndarray
def measure(arr, norm=1):
  if np.size(arr) == 0:
    return 0.0
  if norm == 0:
    return np.count_nonzero(arr)
  if norm == 1:
    return np.sum(np.abs(arr))
  if norm == 2:
    return np.sum(np.square(arr)) # add np.abs on inside to use with complex numbers
  if norm == np.inf:
    return np.max(np.abs(arr))

def measure_dict_vect(vect, norm=1):
  return measure(list(vect.values()))

def measure_dict_mat(matrix, norm=1):
  magnitude = 0.0
  for key in matrix.keys():
    magnitude += measure_dict_vect(matrix[key], norm)
  if norm == 0:
    return int(magnitude)
  return magnitude

## Inverse Methods

In [ ]:
# @title Gaussian Elimination (adjacency list)
# source: https://www.javatpoint.com/gaussian-elimination-in-python  (modified to work with adjacency lists)
import copy
def gaussian_inv_dict(M_dict, j, verbose=0):
  M = copy.deepcopy(M_dict)
  size = len(list(M.keys()))  # assume it is invertible, so every column has nonzero entries
  M[size] = {j:1}

  FLOPs = 0

  for pivot in range(0, size):
    # Searching the maximum value in the pivot column (don't count these FLOPs as a smarter method could do this live)
    max_el = 0
    max_row = pivot
    for row in M[pivot].keys():
      if row < pivot:
        continue
      if abs(M[pivot][row]) > max_el:
        max_el = abs(M[pivot][row])
        max_row = row

    # make the max row the pivot row (don't count these FLOPs as a smarter method could alias the operations in-place)
    for col in range(pivot, size+1):
      if max_row in M[col].keys():
        temp = M[col][max_row]
      else:
        temp = 0
      if pivot in M[col].keys():
        M[col][max_row] = M[col][pivot]
      else:
        if max_row in M[col].keys():
          del M[col][max_row]
      if temp != 0:
        M[col][pivot] = temp
      else:
        if pivot in M[col].keys():
          del M[col][pivot]

    # scale the pivot row to have 1-diagonal
    for col in range(size, pivot-1, -1):
      if pivot in M[col].keys():
        M[col][pivot] /= M[pivot][pivot]
        FLOPs += 1 # for division

    # Changing the value of the rows other than the diagonal (pivot) to 0 in the pivot column
    for row in M[pivot].keys():
      if row == pivot: # don't touch the pivot row
        continue

      leading = M[pivot][row]

      for col in range(size, pivot-1, -1):
        if row not in M[col].keys():
          M[col][row] = 0.0
        if pivot in M[col].keys():
          M[col][row] -= leading * M[col][pivot] # other rows, subtract pivot row scaled to make pivot column zero
          FLOPs += 2 # multiplication and subtraction

  return M[size], FLOPs

In [ ]:
# @title Power series estimate of inverse column (array)
def pow_estimate(M, j, iterations=10, epsilon=10**-3, verbose=0):
  size = np.shape(M)[0]
  u = elem_vect(size, j)
  estimate = u

  Q = np.identity(size) - M
  for iter in range(iterations): # just computes jth column of I + Q + Q^2 + Q^3 + ... + Q^(iterations-1)

    # get the current power term in O(p^2)
    u = np.matmul(Q, u)

    for i in range(size):
      # round if close to zero (this is neccesary to limit our search to j's epsilon-horizon; comment it out for an exact solution) in O(1) each p times
      if abs(u[i]) < epsilon:
        u[i] = 0.0

    # add it to the power sum in O(p)
    estimate = estimate + u

  return estimate # in O(iters * n^2)

In [ ]:
# @title Power series estimate of inverse column (adjacency list)
def pow_estimate_dict(M, j, iterations=10, epsilon=10**-3, verbose=0):
  # where M[col][row]
  def Q(row, col): # Q = I - M
    return (row == col) - M[col][row]

  u = {} # u ~ M^-1e_j = (I-Q)^-1e_j = e_j + Qe_j + Q^2e_j + Q^3e_j + Q^4e_j + ...
  curr = {j:1} # e_j

  FLOPs = 0

  for iter in range(iterations):
    if verbose > 1:
      print(dict_mat_mult(M,u,epsilon))

    # save the flow to the current estimate - the column of M equals the row of u in the multiplication
    u, flops = add_vects(u, curr)
    FLOPs += flops

    # compute the new fringe flows Q^(k+1)e_j = Q * (Q^k)e_j
    curr, flops = mul_Q_left(M, curr)
    FLOPs += flops

    magnitude = measure(list(curr.values()))
    if verbose > 0 and iter%5 == 0:
      print(f"||Q^k e_{j}|| = {magnitude:.4f}")
    if magnitude < epsilon:
      break

  return u, FLOPs # in O(p^2)

In [ ]:
# @title Priority Queue (adjacency list)
from queue import PriorityQueue
from math import copysign
def queue_estimate_dict(M, j, iterations=10, epsilon=10**-3, verbose=0):
  u = {} # u ~ M^-1 e_j = (I-Q)^-1 e_j = e_j + Qe_j + Q^2 e_j + Q^3 e_j + Q^4 e_j + ...

  # start with the flow from j on the first path step (one unit flows into the netwoek at j)
  queue = PriorityQueue()
  queue.put((-1.0, False, j, 0))

  FLOPs = 0

  for iter in range(iterations):
    if verbose > 0:
      print(queue.queue)

    if queue.empty():
      break

    # get the most significant flow for the next round
    neg_flow_mag, is_neg, col, depth = queue.get()

    # save the flow to the current estimate
    FLOPs += 1 # moving node from fringe
    if col not in u.keys(): # the column of M equals the row of u in the multiplication
      u[col]  = -neg_flow_mag * (-1)**is_neg
    else:
      u[col] += -neg_flow_mag * (-1)**is_neg

    # compute the new fringe flows and add them to the queue
    if -neg_flow_mag > epsilon:
      for row in M[col].keys():
        FLOPs += 1 # for each new flow path
        queue.put((abs(get_Q(M,row,col)) * neg_flow_mag, is_neg ^ (get_Q(M,row,col) < 0), row, depth+1)) # (flow, row/node)

  return u, FLOPs # in O(p)?

In [ ]:
# @title Power series in epsilon horizon (dynamic priority queue) (adjacency list)
def pow_estimate_epsilon_dict(M, j, iterations=10, epsilon=10**-3, verbose=0):
  u = {} # u ~ M^-1e_j = (I-Q)^-1e_j = e_j + Qe_j + Q^2e_j + Q^3e_j + Q^4e_j + ...
  curr = {j:1} # e_j

  FLOPs = 0

  for iter in range(iterations):
    if verbose > 1:
      print(dict_mat_mult(M,u,epsilon))

    # save the flow to the current estimate - the column of M equals the row of u in the multiplication
    u, flops = add_vects(u, curr)
    FLOPs += flops

    # compute the new fringe flows Q^(k+1)e_j = Q * (Q^k)e_j
    curr, flops = mul_Q_left(M, curr, epsilon)
    FLOPs += flops

    magnitude = measure_dict_vect(curr)
    if verbose > 0 and iter%5 == 0:
      print(f"||Q^k e_{j}|| = {magnitude:.4f}")
    if magnitude < epsilon:
      break

  return u, FLOPs # in O(p^2)

In [ ]:
# @title Power series Pick-up (adjacency list)
def recover_power_series(M, u={}, curr=None, last_term=None, iterations=10, epsilon=10**-3, verbose=0):
  # u ~ previous + sum(Q^k * old remainder) + sum(k * Q^k * last term)
  if curr is None:
    curr = {j:1} # e_j
  remainder = {}

  FLOPs = 0

  for iter in range(iterations):
    if verbose > 1:
      print(dict_mat_mult(M,u,epsilon))

    # save the flow to the current estimate - the column of M equals the row of u in the multiplication
    u, flops = add_vects(u, curr)
    FLOPs += flops

    # compute the new fringe flows Q^(k+1)e_j = Q * (Q^k)e_j
    curr, flops = mul_Q_left(M, curr)
    FLOPs += flops

    magnitude = measure_dict_vect(curr)
    if verbose > 0 and iter%5 == 0:
      print(f"||Q^k e_{j}|| = {magnitude:.4f}")
    if magnitude < epsilon:
      break

    for key in list(curr.keys()):
      if curr[key] < epsilon:
        if key not in remainder:
          remainder[key] = 0.0
        remainder[key] += curr[key]
        del curr[key]

  return u, remainder, curr, FLOPs # in O(p^2)

In [ ]:
# @title Feedback estimate of inverse column (array)
def ml_estimate(M, j, iterations=10, epsilon=10**-3, learning_rate=1, verbose=0):
  size = np.shape(M)[0]

  # start with the minimum input needed to measure the effect of j on i - the flow out from j (this is not related to the target being this vector) in O(1)
  u = elem_vect(size, j)

  for iter in range(iterations):
    # get the current prediction in O(p^2)
    y = np.matmul(M,u)

    # easy part - scale the output to make the jth entry == 1 in O(p)
    if abs(y[j]) > epsilon:
      u = u / y[j]
      y = y / y[j]

    for i in range(size):

      # remove enough so non-j y tends to zero when scaled by the through-flow at node i (doesn't account for consequent flow effects) in O(p)
      if i != j and abs(M[i,i]) > epsilon:
        u[i] -= learning_rate * y[i] / M[i,i] # /y[j] was here, but is 1 unless y[j] was close to epsilon in O(1) each p times

      # round if close to zero (this is neccesary to limit our search to j's epsilon-horizon) in O(1) each p times
      if abs(u[i]) < epsilon:
        u[i] = 0.0

  return u # in O(iters * p^2)


In [ ]:
# @title Feedback estimate of inverse column (adjacency list)
def ml_estimate_dict(M, j, iterations=10, epsilon=10**-3, learning_rate=1.0, verbose=0):

  # start with the minimum input needed to measure the effect of j on i - the flow out from j (this is not related to the target being this vector) in O(1)
  u = {j: 1} # equivalent of elem_vect(size, j)

  FLOPs = 0

  for iter in range(iterations):
    # get the current prediction in O(p^2)
    y, flops = dict_mat_mult(M, u)
    FLOPs += flops

    # easy part - scale the output to make the jth entry == 1 in O(p)
    if j in y.keys() and abs(y[j]) > epsilon:
      scale = y[j]
    else:
      scale = 1.0

    for row in u.keys():
      FLOPs += 1 # normalization
      u[row] = u[row]/scale # scale the output later, input now (if we scale the input later, some entries won't get scaled, as the matchin entry in y has no key (is zero))

    # use each output error y - e_j to inform the input u
    stop_early = True
    for row in y.keys():
      if abs((row == j) - y[row]) > epsilon:
        stop_early = False

      if row not in u.keys():
        u[row] = 0.0

      # remove enough so non-j y tends to zero when scaled by the through-flow at node i (doesn't account for consequent flow effects) in O(p)
      if row != j and row in M.keys() and row in M[row].keys() and abs(M[row][row]) > epsilon:
        FLOPs += 1
        u[row] -= learning_rate * y[row]/scale / M[row][row] # no need to scale the output until now

      # round if close to zero (this is neccesary to limit our search to j's epsilon-horizon) in O(1) each p times
      if abs(u[row]) < epsilon:
        del u[row]

    # don't do any more iterations if the estimate is good
    if stop_early:
      break

  return u, FLOPs # in O(iters * p^2)

# Testing

## Generate the matrix and set paramters

In [ ]:
# @title Set hyperparameters
use_array = False

j = 1 # source node
i = 150 # sink node

precision = 2 # int, min value is 1
epsilon = 10**(-precision)
iterations = 1000
learning_rate = 1

n = 2 # the number of nodes in the network
k = 2 # the largest allowable sized SCC in the network
spectral_radius = 1 - epsilon

sparsity = 0.9 # the proportion of forward edges that are zero, between 0 and 1

In [ ]:
# @title Generate the matrices and set values to use later
if use_array:
  Q = setTotalColumnValueToOne(generateBLTmatrix(n, k, sparsity)) * (spectral_radius)
  M = np.identity(n) - Q
  M_dict = matrixToDict(M)
else:
  M_dict = generateSparseAdjacencyList(n,k,0.6)

## Compute the Inverse Column

* exact_n
* pow_est_n
* queue_dict_est_n
* ml_arr_est_n
* ml_dict_est_n

In [ ]:
#@title Numpy Matrix Inversion
if use_array:
  start = time.time()
  exact_n = np.linalg.inv(M)[j,:]
  exact_time = time.time() - start

In [ ]:
#@title Power Series Estimate (array)
if use_array:
  start = time.time()
  pow_est_n = pow_estimate(M, j=j, iterations=iterations, epsilon=epsilon, verbose=0)
  pow_time = time.time() - start

In [ ]:
#@title Feedback estimate (array)
if use_array:
  start = time.time()
  ml_arr_est_n = ml_estimate(M, j=j, iterations=iterations, epsilon=epsilon, learning_rate=learning_rate, verbose=0)
  arr_time = time.time() - start

In [ ]:
#@title Gaussian Elimination (adjacency list)
start = time.time()
gauss_n, gauss_flops = gaussian_inv_dict(M_dict, j)
if not use_array:
  exact_n = dict_to_vect(gauss_n,n)
gauss_time = time.time() - start

In [ ]:
#@title Power Series Estimate (adjacecny list)
start = time.time()
pow_dict_est_n, pow_flops = pow_estimate_dict(M_dict, j=j, iterations=iterations, epsilon=epsilon, verbose=0)
pow_dict_time = time.time() - start

In [ ]:
#@title Priority estimate (adjacency list)
start = time.time()
queue_dict_est_n, queue_flops = queue_estimate_dict(M_dict, j=j, iterations=iterations, epsilon=epsilon, verbose=0)
queue_time = time.time() - start

In [ ]:
#@title Feedback estimate (adjacency list)
start = time.time()
ml_dict_est_n, ml_flops = ml_estimate_dict(M_dict, j=j, iterations=iterations, epsilon=epsilon, learning_rate=learning_rate, verbose=0)
dict_time = time.time() - start

## Display the Results

In [ ]:
if use_array:
  print(f"The exact solution took {exact_time:.4f} sec (Numpy)")
print(f"The exact solution took {gauss_time if use_array else gauss_flops:.4f} sec (adjacency list)")
print(f"At {iterations} iterations with epsilon={epsilon}, the preimage errors were:")
if use_array:
  print_errors_array(pow_est_n, exact_n, "Power Series Array", pow_time)
  print_errors_array(ml_arr_est_n, exact_n, "Machine Learning Array", arr_time)
print_errors_dict(pow_dict_est_n, vectToDict(exact_n), "Power Series Dict", pow_dict_time if use_array else pow_flops)
print_errors_dict(queue_dict_est_n, vectToDict(exact_n), "Priority Queue Dict", queue_time if use_array else queue_flops)
print_errors_dict(ml_dict_est_n, vectToDict(exact_n), "Machine Learning Dict", dict_time if use_array else ml_flops)
print(f"And the postimage errors* were:")
if use_array:
  e_j = elem_vect(n,j)
  print_errors_array(np.matmul(M,pow_est_n), e_j, "Power Series Array", pow_time)
  print_errors_array(np.matmul(M,ml_arr_est_n), e_j, "Machine Learning Array", arr_time)
print_errors_dict(dict_mat_mult(M_dict, pow_dict_est_n)[0], {j:1}, "Power Series Dict", pow_dict_time if use_array else pow_flops)
print_errors_dict(dict_mat_mult(M_dict, queue_dict_est_n)[0], {j:1}, "Priority Queue Dict", queue_time if use_array else queue_flops)
print_errors_dict(dict_mat_mult(M_dict, ml_dict_est_n)[0], {j:1}, "Machine Learning Dict", dict_time if use_array else ml_flops)
print("*Note: The postimage is probably exact after rounding by epsilon, so we display its error without rounding.")

The exact solution took 12.0000 sec (adjacency list)
At 1000 iterations with epsilon=0.01, the preimage errors were:
  For the Power Series Dict method (57.0000 sec):
    Act L0:   2
    Avg L0:   1.0
    Avg L1:   0.007558271999999894
    Avg L2:   0.008165134779563167
    Avg Linf: 0.010647276749502499
  For the Priority Queue Dict method (65.0000 sec):
    Act L0:   2
    Avg L0:   1.0
    Avg L1:   0.05775532638981934
    Avg L2:   0.05985755092139703
    Avg Linf: 0.07347946947976469
  For the Machine Learning Dict method (17.0000 sec):
    Act L0:   0.0
    Avg L0:   0.0
    Avg L1:   0.0
    Avg L2:   0.0
    Avg Linf: 0.0
And the postimage errors* were:
  For the Power Series Dict method (57.0000 sec):
    Act L0:   2
    Avg L0:   1.0
    Avg L1:   0.0030233087999999436
    Avg L2:   0.0032668350107765863
    Avg Linf: 0.004260973895039211
  For the Priority Queue Dict method (65.0000 sec):
    Act L0:   2
    Avg L0:   1.0
    Avg L1:   0.02310213055592765
    Avg L2:   0.023

# **Next Steps:**
* Make multi-trial benchmarking automatic
* Make the precision adjust dynamically per iteration so we get better and better results (an option in the arguments, like tempurature rate)
* normalize the columns as we read them to sum to less than 1 (make it stochastic so the numerical method works better) and sawp coluns to make the diagonal of Q equal to zero
* save the rounded epsilon paths in pow_estimate_epsilon_dict in a seperate double map so we can increase the accuracy if needed. Pass this and the current estimate into the method and use it as the starting point (with default values e_j and the 0 estimate)

In [ ]:
n = 20
k = 5
epsilon = 10**-5
iterations = 100
learning_rate = 1

M_dict = generateSparseAdjacencyList(n,k,0.6)
if n < 7:
  print(M_dict)
M = dict_to_mat(M_dict, n)
Q = np.identity(n) - M
u,s,v_t = np.linalg.svd(Q)
print("spectral radius is:", max(abs(s)))
print(f"Diagonal of Q is on the interal [{np.min(np.abs(np.diagonal(Q))):.4f}, {np.max(np.abs(np.diagonal(Q))):.4f}] (must be in (-1,1))")

spectral radius is: 0.6246941425142074
Diagonal of Q is on the interal [0.1252, 0.2699] (must be in (-1,1))


In [ ]:
def use_method(M_dict, size, method, **kwargs):
  M_inv = {}
  FLOPs = 0.0

  for col in range(size):
    u, flops = method(M_dict, col, **kwargs)
    FLOPs += float(flops) / size
    M_inv[col] = u

  M_inv = dict_to_mat(M_inv, size)
  L1_error =  measure(dict_to_mat(M_dict, size) @ M_inv - np.identity(size))
  return FLOPs, L1_error, M_inv

In [ ]:
FLOPs_gauss, L1_error_gauss, M_inv_gauss = use_method(M_dict, n, gaussian_inv_dict) # 21-37 sec for n=100, k=10
print("FLOPs:", FLOPs_gauss)
print("Error:", L1_error_gauss)

FLOPs: 5140.75
Error: 5.376685308739606e-15


In [ ]:
FLOPs_pow, L1_error_pow, M_inv_pow = use_method(M_dict, n, pow_estimate_dict, iterations=iterations, epsilon=epsilon, verbose=0) # 4 sec for n=100, k=10

print("FLOPs:", FLOPs_pow)
print("% of Gaussian Elimination:", FLOPs_pow/FLOPs_gauss)
print("Error:", L1_error_pow)

FLOPs: 2747.6000000000004
% of Gaussian Elimination: 0.5344745416524826
Error: 0.00015794604461065898


In [ ]:
FLOPs_queue, L1_error_queue, M_inv_queue = use_method(M_dict, n, queue_estimate_dict, iterations=10000, epsilon=epsilon, verbose=0) # 34-36 sec for n=100, k=10 BUT DIVERGES

print("FLOPs:", FLOPs_queue)
print("% of Gaussian Elimination:", FLOPs_queue/FLOPs_gauss)
print("Error:", L1_error_queue)

FLOPs: 29514.0
% of Gaussian Elimination: 5.741185624665662
Error: 0.6267769679607347


In [ ]:
FLOPs_pow_ep, L1_error_pow_ep, M_inv_pow_ep = use_method(M_dict, n, pow_estimate_epsilon_dict, iterations=iterations, epsilon=epsilon, verbose=0) # 1 sec for n=100, k=10

print("FLOPs:", FLOPs_pow_ep)
print("% of Gaussian Elimination:", FLOPs_pow_ep/FLOPs_gauss)
print("Error:", L1_error_pow_ep)

FLOPs: 1675.55
% of Gaussian Elimination: 0.32593493167339394
Error: 0.013713624828245426


In [ ]:
M_inv_pickup = {}
FLOPs_pickup = 0.0

for col in range(n):
  u, remainder, last_term, flops = recover_power_series(M_dict, u={}, curr={j:1}, last_term=None, iterations=iterations, epsilon=epsilon, verbose=0)
  FLOPs_pickup += float(flops) / n
  M_inv_pickup[col] = u

# M_inv_pickup = {}
# FLOPs_pickup = 0.0

# for col in range(n):
#   u, remainder, seed, flops = recover_power_series(M_dict, u=u, seed=remainder, last_term=last_term, iterations=iterations, epsilon=epsilon**2, verbose=0)
#   FLOPs_pickup += float(flops) / n
#   M_inv_pickup[col] = u

M_inv_pickup = dict_to_mat(M_inv_pickup, n)
L1_error_pickup =  measure(dict_to_mat(M_dict, n) @ M_inv_pickup - np.identity(n))

print("FLOPs:", FLOPs_pickup)
print("% of Gaussian Elimination:", FLOPs_pickup/FLOPs_gauss)
print("Error:", L1_error_pickup)

FLOPs: 1951.9999999999993
% of Gaussian Elimination: 0.3797111316442152
Error: 38.00308849364763


In [ ]:
FLOPs_ml, L1_error_ml, M_inv_ml = use_method(M_dict, n, ml_estimate_dict, iterations=iterations, learning_rate=learning_rate, epsilon=epsilon, verbose=0) # 1-2 sec for n=100, k=10

print("FLOPs:", FLOPs_ml)
print("% of Gaussian Elimination:", FLOPs_ml/FLOPs_gauss)
print("Error:", L1_error_ml)


FLOPs: 1683.0
% of Gaussian Elimination: 0.32738413655595
Error: 0.00074809855230711


In [ ]:
import pandas as pd
data_frame = pd.DataFrame(columns = ['FLOPs', '% of Gaussian Elimination', 'Error'])
#data_frame.columns = ['FLOPs', '% of Gaussian Elimination', 'Error']

newrow = [FLOPs_gauss, 1, L1_error_gauss]
data_frame.loc[len(data_frame.index)] = newrow

newrow = [FLOPs_pow, FLOPs_pow/FLOPs_gauss, L1_error_pow]
data_frame.loc[len(data_frame.index)] = newrow

newrow = [FLOPs_queue, FLOPs_queue/FLOPs_gauss, L1_error_queue]
data_frame.loc[len(data_frame.index)] = newrow

newrow = [FLOPs_pow_ep, FLOPs_pow_ep/FLOPs_gauss, L1_error_pow_ep]
data_frame.loc[len(data_frame.index)] = newrow

newrow = [FLOPs_pickup, FLOPs_pickup/FLOPs_gauss, L1_error_pickup]
data_frame.loc[len(data_frame.index)] = newrow

newrow = [FLOPs_ml, FLOPs_ml/FLOPs_gauss, L1_error_ml]
data_frame.loc[len(data_frame.index)] = newrow

data_frame.index = ['Gauss', 'Pow', 'Queue', 'Poew_ep', 'Pickup', 'ml']
display(data_frame)

,FLOPs,% of Gaussian Elimination,Error
Gauss,5140.75,1.000000,5.376685e-15
Pow,2747.60,0.534475,1.579460e-04
Queue,29514.00,5.741186,6.267770e-01
Poew_ep,1675.55,0.325935,1.371362e-02
Pickup,1952.00,0.379711,3.800309e+01
ml,1683.00,0.327384,7.480986e-04


Notes:
* Q's columns must be $||\cdot||_1 < 1$, not necesarily M's
* Q's diagonal must be on the interval (-1,1), the closer to 0 the better
 * so M's diagonal must be on (0,2) but closer to 1 is best